In [ ]:
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

from scipy.stats import norm

## Swain vs. Alabama ##

In [ ]:
# SD of population is sqrt(p*(1-p))

SD_pop = (0.26 * (1 - 0.26)) ** 0.5
SD_pop

In [ ]:
SD_panels_formula = SD_pop / (100) ** 0.5
SD_panels_formula

In [ ]:
x1 = np.arange(0.055,.40, 0.001)
plots.plot(x1, norm.pdf(x1, 0.26, SD_panels_formula));
plots.axvline(0.08, color='red', lw=2);
print("p-value from the normal distribution = ", norm.cdf(0.08, loc = 0.26, scale =  SD_panels_formula))

In [ ]:
z_score = (0.08 - 0.26) / SD_panels_formula
z_score

In [ ]:
x2 = np.arange(-5,5, 0.1)
plots.plot(x2, norm.pdf(x2, 0, 1));
plots.axvline(z_score, color='red', lw=2);

In [ ]:
norm.cdf(z_score)

# Kristen Gilbert

### Construct table of the data

In [ ]:
Gilbert_present = np.repeat(['Yes', 'No'], [257, 1384])

Deaths = np.repeat([1, 0, 1, 0], [40, 217, 34, 1350])

Gilbert = Table().with_columns(
    'Gilbert present', Gilbert_present,
    'Deaths', Deaths
)

### Determine observed difference

In [ ]:
def difference_of_means(table, label, group_label):
    """Takes: name of table, column label of numerical variable,
    column label of group-label variable
    Returns: Difference of means of the two groups"""
    
    #table with the two relevant columns
    reduced = table.select(label, group_label)  
    
    # table containing group means
    means_table = reduced.group(group_label, np.average)
    
    # array of group means
    means = means_table.column(1)
    
    return means.item(0) - means.item(1)

In [ ]:
obs_diff_prop = difference_of_means(Gilbert, 'Deaths', 'Gilbert present')
obs_diff_prop

### Simulate one outcome

In [ ]:
def one_simulated_difference(table, outcome, group_label):
    """Takes: name of table, column label of binary cateogorcial variable,
    column label of group-label variable
    Returns: Difference of proportions of the two groups after shuffling labels"""
    
    # array of shuffled labels
    shuffled_labels = table.sample(with_replacement = False).column(group_label)
    
    # table of outcome variable and shuffled labels
    shuffled_table = table.select(outcome).with_column(
        'Shuffled Label', shuffled_labels)
    
    return difference_of_means(shuffled_table, outcome, 'Shuffled Label')  

### Simulate many outcomes

In [ ]:
differences = make_array()

for i in np.arange(1000):
    new_difference = one_simulated_difference(Gilbert, 'Deaths', 'Gilbert present')
    differences = np.append(differences, new_difference)

### Use simulation to make a decision

In [ ]:
Table().with_column('Difference Between Group Proportions', differences).hist(bins = np.arange(-0.05, 0.05, 0.005))
print('Observed Difference:', obs_diff_prop)
plots.title('Prediction Under the Null Hypothesis');
plots.axvline(obs_diff_prop, color='red', lw=2);
plots.xticks(rotation = 270);

In [ ]:
p_value = sum(differences <= obs_diff_prop)/1000
p_value

In [ ]:
SD_shifts = np.std(differences)
SD_shifts

In [ ]:
Table().with_column('Difference Between Group Proportions', differences).hist(bins = np.arange(-0.05, 0.05, 0.005))
print('Observed Difference:', obs_diff_prop)
plots.title('Prediction Under the Null Hypothesis');
plots.axvline(obs_diff_prop, color='red', lw=2);
plots.xticks(rotation = 270);

x3 = np.arange(-0.14, 0.05, 0.0005)
plots.plot(x3, norm.pdf(x3, 0, SD_shifts));

### Make a Decision *without* Simulation

In [ ]:
pooled_prop = np.mean(Gilbert.column('Deaths'))
pooled_prop

In [ ]:
shifts_with_death_sd = np.sqrt(pooled_prop * (1 - pooled_prop) * (1/257 + 1/1384))
shifts_with_death_sd

In [ ]:
Table().with_column('Difference Between Group Proportions', differences).hist(bins = np.arange(-0.05, 0.05, 0.005))
print('Observed Difference:', obs_diff_prop)
plots.title('Prediction Under the Null Hypothesis');
plots.axvline(obs_diff_prop, color='red', lw=2);
plots.xticks(rotation = 270);

x3 = np.arange(-0.14, 0.05, 0.0005)
plots.plot(x3, norm.pdf(x3, 0, shifts_with_death_sd));

In [ ]:
norm.cdf(obs_diff_prop, 0, shifts_with_death_sd)

### Dolphin Therapy

In [ ]:
Therapy = np.repeat(['Dolphin', 'No Dolphin'] , [13, 12])

Improvement = np.repeat([1, 0, 1, 0], [10, 3, 3, 9])

dolphin = Table().with_columns(
    'Therapy', Therapy,
    'Improvement', Improvement
)

In [ ]:
obs_diff_dolphin_prop = difference_of_means(dolphin, 'Improvement', 'Therapy')
obs_diff_dolphin_prop

## Simulated Dolphin Therapy

In [ ]:
dolphin_differences = make_array()

for i in np.arange(10000):
    new_dolphin_difference = one_simulated_difference(dolphin, 'Improvement', 'Therapy')
    dolphin_differences = np.append(dolphin_differences, new_dolphin_difference)

In [ ]:
Table().with_column('Difference Between Group Proportions', dolphin_differences).hist(bins = 11)
print('Observed Difference:', obs_diff_dolphin_prop)
plots.title('Prediction Under the Null Hypothesis');
plots.axvline(obs_diff_dolphin_prop, color='red', lw=2);

In [ ]:
p_value = sum(dolphin_differences >= obs_diff_dolphin_prop)/10000
p_value

In [ ]:
pooled_dolphin_prop = np.mean(dolphin.column('Improvement'))

dolphin_improv_sd = np.sqrt(pooled_dolphin_prop * (1 - pooled_dolphin_prop) * (1/13 + 1/12))
dolphin_improv_sd

In [ ]:
Table().with_column('Difference Between Group Proportions', dolphin_differences).hist()
print('Observed Difference:', obs_diff_dolphin_prop)
plots.title('Prediction Under the Null Hypothesis');
plots.axvline(obs_diff_dolphin_prop, color='red', lw=2);

x4 = np.arange(-0.8,.8, 0.005)
plots.plot(x4, norm.pdf(x4, 0, dolphin_improv_sd));

In [ ]:
norm.cdf(obs_diff_dolphin_prop, 0, dolphin_improv_sd)

In [ ]:
1 - norm.cdf(obs_diff_dolphin_prop, 0, dolphin_improv_sd)

In [ ]:
Table().with_column('Difference Between Group Proportions', dolphin_differences).hist(bins = 100)
print('Observed Difference:', obs_diff_dolphin_prop)
plots.title('Prediction Under the Null Hypothesis');
plots.axvline(obs_diff_dolphin_prop, color='red', lw=2);